# Extract KG Triples from `real_contradiction`

Two-stage extraction pipeline over the 131-pair real_contradiction corpus:

1. **Open extraction**: LLM picks predicates freely; we canonicalize post-hoc.
2. **Derive closed schema** from the top-K most frequent predicates produced by the open stage.
3. **Closed-schema extraction**: LLM is restricted to that derived vocabulary.
4. **Hallucination pruning** (van Cauter & Yakovets, 2024): drop any triple whose subject or object is absent from the source sentence, or whose predicate is not in the schema.

Outputs are saved to `data/processed/kg_triples/`.

In [52]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from openai import OpenAI
from pydantic_settings import BaseSettings, SettingsConfigDict
from tqdm import tqdm

In [41]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )
    openai_api_key: str
    llm_model: str = "gpt-5.4-mini"


settings = Settings()
client = OpenAI(api_key=settings.openai_api_key)
OUTPUT_DIR = Path("data/processed/kg_triples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using model: {settings.llm_model}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Using model: gpt-5.4-mini
Output directory: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\kg_triples


In [42]:
df = pd.read_csv("data/processed/finding_contradictions_in_text_2008/real_contradiction.csv")
print(f"Loaded {len(df)} pairs")
df.head()

Loaded 131 pairs


,contradiction,type,t,h
0,YES,negation,Tariq Aziz was not considered a member of Sadd...,Tariq Aziz was in Saddam's inner circle.
1,YES,lexical,Tariq Aziz kept outside the closed circle of S...,Tariq Aziz was in Saddam's inner circle.
2,YES,negation,Tariq Aziz was not one of the most powerful fi...,Tariq Aziz was prominent in the regime.
3,YES,lexical,Tariq Aziz retained influence.,Aziz had virtually no power.
4,YES,numeric,Alexandros Yiotopoulos is 58.,Alexandros Yiotopoulos is 59 years old.


## Shared extraction helpers

`extract_triples()` calls OpenAI in JSON mode with a supplied prompt. `canonicalize()` normalizes predicate surface forms for the closed schema.

In [43]:
OPEN_PROMPT = (
    "Extract knowledge graph triples from the following sentence.\n\n"
    "Rules:\n"
    "- DECOMPOSE the sentence into multiple atomic triples. Each triple captures "
    "ONE simple (subject, predicate, object) fact. Prefer MANY short triples over "
    "ONE triple with a long object.\n"
    "- Each subject and each object must be a single entity expressed as a short "
    "noun phrase (1-3 words ideally). Strip articles (a/an/the), relative clauses, "
    "and modifiers. Do NOT pack a whole clause into the object.\n"
    "- Example of good decomposition:\n"
    '    Sentence: "Tariq Aziz kept outside Saddam\'s inner circle of Sunni cronies."\n'
    "    Triples: [(Tariq Aziz, kept_outside, inner circle),\n"
    "              (inner circle, belongs_to, Saddam),\n"
    "              (inner circle, described_as, Sunni cronies)]\n"
    "  BAD (one triple with long object):\n"
    '    [(Tariq Aziz, kept_outside, "Saddam\'s inner circle of Sunni cronies")]\n'
    "- Preserve negation inside the predicate. If a claim is explicitly negated "
    "(not, never, no, refused to, failed to, etc.), prefix the predicate with 'not '. "
    "Do NOT bury negation in the object.\n\n"
    "Return strict JSON of the form: "
    '{{"triples": [{{"subject": "...", "predicate": "...", "object": "..."}}]}}\n\n'
    "Sentence: {sentence}"
)


def extract_triples(sentence: str, prompt_template: str, **kwargs) -> list[dict]:
    prompt = prompt_template.format(sentence=sentence, **kwargs)
    try:
        response = client.chat.completions.create(
            model=settings.llm_model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        payload = json.loads(response.choices[0].message.content)
        triples = payload.get("triples", [])
        return [t for t in triples if isinstance(t, dict)]
    except Exception as e:
        print(f"Extraction failed for sentence: {sentence[:60]}... | {e}")
        return []


def canonicalize(pred: str | None) -> str | None:
    if pred is None:
        return None
    return pred.strip().lower().replace(" ", "_").replace("-", "_")

## Smoke test on first 3 pairs (open extraction)

Confirms the API call and JSON parsing work before committing to 131 pairs.

In [44]:
for idx, row in enumerate(df.head(3).itertuples(index=False), start=1):
    print(f"[id={idx}] type={row.type}")
    print(f"  t: {row.t}")
    print(f"  h: {row.h}")
    open_t = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in extract_triples(row.t, OPEN_PROMPT)]
    open_h = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in extract_triples(row.h, OPEN_PROMPT)]
    print(f"  open t: {open_t}")
    print(f"  open h: {open_h}")
    print("─" * 100)

[id=1] type=negation
  t: Tariq Aziz was not considered a member of Saddam's innermost circle.
  h: Tariq Aziz was in Saddam's inner circle.
  open t: [{'subject': 'Tariq Aziz', 'predicate': 'not_considered', 'object': 'member'}, {'subject': 'member', 'predicate': 'of', 'object': 'Saddam'}, {'subject': 'member', 'predicate': 'described_as', 'object': 'innermost circle'}]
  open h: [{'subject': 'Tariq Aziz', 'predicate': 'was_in', 'object': 'inner circle'}, {'subject': 'inner circle', 'predicate': 'belongs_to', 'object': 'Saddam'}]
────────────────────────────────────────────────────────────────────────────────────────────────────
[id=2] type=lexical
  t: Tariq Aziz kept outside the closed circle of Saddam's Sunni Moslem cronies.
  h: Tariq Aziz was in Saddam's inner circle.
  open t: [{'subject': 'Tariq Aziz', 'predicate': 'kept_outside', 'object': 'circle'}, {'subject': 'circle', 'predicate': 'described_as', 'object': 'closed'}, {'subject': 'circle', 'predicate': 'belongs_to', 'object

## Open extraction on all 131 pairs

Appends to JSONL after each pair so a partial run doesn't lose work.

In [45]:
open_file = OUTPUT_DIR / "open_extraction.jsonl"

with open_file.open("w", encoding="utf-8") as f:
    for idx, row in enumerate(tqdm(list(df.itertuples(index=False)), desc="Open"), start=1):
        t_raw = extract_triples(row.t, OPEN_PROMPT)
        h_raw = extract_triples(row.h, OPEN_PROMPT)
        t_triples = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in t_raw]
        h_triples = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in h_raw]
        record = {
            "id": str(idx),
            "type": row.type,
            "t_text": row.t,
            "h_text": row.h,
            "t_triples": t_triples,
            "h_triples": h_triples,
            "notes": "",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

print(f"Open extraction done -> {open_file}")

Open: 100%|██████████| 131/131 [05:58<00:00,  2.73s/it]

Open extraction done -> data\processed\kg_triples\open_extraction.jsonl


## Derive closed schema from the open-extraction output

Count canonicalized predicates across all open-extraction triples, pick the top `TOP_K` as the closed vocabulary. Adjust `TOP_K` to control the trade-off between coverage and schema tightness.

In [46]:
TOP_K = 25


def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


open_records = load_jsonl(open_file)
predicate_counts = Counter(tr.get("predicate") for r in open_records for tr in r["t_triples"] + r["h_triples"] if tr.get("predicate"))

print(f"Unique predicates from open extraction: {len(predicate_counts)}")
print(f"Total predicate occurrences: {sum(predicate_counts.values())}")
print(f"\nTop {TOP_K} predicates:")
for pred, count in predicate_counts.most_common(TOP_K):
    print(f"  {pred}: {count}")

PREDICATES = [pred for pred, _ in predicate_counts.most_common(TOP_K)]
print(f"\nDerived closed schema ({len(PREDICATES)} predicates):")
print(PREDICATES)

Unique predicates from open extraction: 544
Total predicate occurrences: 1118

Top 25 predicates:
  is: 60
  of: 46
  said: 38
  described_as: 29
  in: 24
  located_in: 21
  was: 20
  for: 19
  occurred_in: 17
  from: 14
  by: 13
  includes: 11
  killed: 9
  to: 9
  caused: 9
  found: 8
  include: 7
  belongs_to: 6
  lasted: 6
  over: 6
  has: 6
  included: 6
  had: 5
  not_arrested_in: 5
  not_allow: 5

Derived closed schema (25 predicates):
['is', 'of', 'said', 'described_as', 'in', 'located_in', 'was', 'for', 'occurred_in', 'from', 'by', 'includes', 'killed', 'to', 'caused', 'found', 'include', 'belongs_to', 'lasted', 'over', 'has', 'included', 'had', 'not_arrested_in', 'not_allow']


## Closed-schema extraction on all 131 pairs

Uses the `PREDICATES` list derived above. The LLM is told to pick only from that list or emit `null`.

In [47]:
CLOSED_PROMPT = (
    "Extract knowledge graph triples from the following sentence.\n\n"
    "Predicates MUST be chosen only from this fixed list:\n{predicates}\n\n"
    "Rules:\n"
    "- DECOMPOSE the sentence into multiple atomic triples. Each triple captures "
    "ONE simple fact. Prefer MANY short triples over ONE triple with a long object.\n"
    "- Each subject and object must be a single entity (1-3-word noun phrase). "
    "Do NOT pack a whole clause into the object.\n"
    "- If no predicate in the list fits a claim, set predicate to JSON null "
    '(literal null, not the string "null" or "None").\n'
    "- Preserve negation in the predicate. If the sentence negates a claim and the "
    "schema contains a matching `not_*` form, use that. Do NOT bury negation inside the object.\n\n"
    "Return strict JSON of the form: "
    '{{"triples": [{{"subject": "...", "predicate": "...", "object": "..."}}]}}\n\n'
    "Sentence: {sentence}"
)

closed_file = OUTPUT_DIR / "closed_schema.jsonl"
predicates_str = ", ".join(PREDICATES)

with closed_file.open("w", encoding="utf-8") as f:
    for idx, row in enumerate(tqdm(list(df.itertuples(index=False)), desc="Closed"), start=1):
        t_triples = extract_triples(row.t, CLOSED_PROMPT, predicates=predicates_str)
        h_triples = extract_triples(row.h, CLOSED_PROMPT, predicates=predicates_str)
        record = {
            "id": str(idx),
            "type": row.type,
            "t_text": row.t,
            "h_text": row.h,
            "t_triples": t_triples,
            "h_triples": h_triples,
            "notes": "",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

print(f"Closed-schema extraction done -> {closed_file}")

Closed: 100%|██████████| 131/131 [05:47<00:00,  2.65s/it]

Closed-schema extraction done -> data\processed\kg_triples\closed_schema.jsonl


## Hallucination pruning (van Cauter & Yakovets, 2024, §3.8)

Canonicalize the extracted triples by removing hallucinated output. A triple is dropped if either:

- its predicate is not in the closed schema, or
- its subject or object does not appear in the source sentence.

Span matching is token-level and case-folded per §3.3: a span is present only when its tokens appear as a contiguous run in the source tokens, so `"filter"` does not match `"filters"`.

In [60]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


def appears_in_tokens(span: str | None, source_tokens: list[str]) -> bool:
    if not span:
        return False
    span_tokens = tokenize(span)
    if not span_tokens:
        return False
    n = len(span_tokens)
    for i in range(len(source_tokens) - n + 1):
        if source_tokens[i : i + n] == span_tokens:
            return True
    return False


def prune_triples(triples: list[dict], source: str, allowed_predicates: set[str]) -> list[dict]:
    source_tokens = tokenize(source)
    kept = []
    for tr in triples:
        if not appears_in_tokens(tr.get("subject"), source_tokens):
            continue
        if not appears_in_tokens(tr.get("object"), source_tokens):
            continue
        if tr.get("predicate") not in allowed_predicates:
            continue
        kept.append(tr)
    return kept


allowed = set(PREDICATES)
closed_pruned_file = OUTPUT_DIR / "closed_schema_pruned.jsonl"
closed_records = load_jsonl(closed_file)

before_total = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in closed_records)

with closed_pruned_file.open("w", encoding="utf-8") as f:
    for r in closed_records:
        r_pruned = {
            **r,
            "t_triples": prune_triples(r["t_triples"], r["t_text"], allowed),
            "h_triples": prune_triples(r["h_triples"], r["h_text"], allowed),
        }
        f.write(json.dumps(r_pruned, ensure_ascii=False) + "\n")

closed_pruned = load_jsonl(closed_pruned_file)
after_total = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in closed_pruned)
dropped = before_total - after_total

print(f"Triples before pruning: {before_total}")
print(f"Triples after pruning:  {after_total}")
print(f"Dropped: {dropped} ({100 * dropped / before_total:.1f}% of original)" if before_total else "Dropped: 0")
print(f"Pruned output -> {closed_pruned_file}")

Triples before pruning: 1226
Triples after pruning:  962
Dropped: 264 (21.5% of original)
Pruned output -> data\processed\kg_triples\closed_schema_pruned.jsonl


## Summary comparison

In [61]:
def coverage(records: list[dict]) -> dict:
    both = sum(1 for r in records if r["t_triples"] and r["h_triples"])
    t_has = sum(1 for r in records if r["t_triples"])
    h_has = sum(1 for r in records if r["h_triples"])
    return {"total": len(records), "t_has_triples": t_has, "h_has_triples": h_has, "both": both}


def count_triples(records: list[dict]) -> int:
    return sum(len(r["t_triples"]) + len(r["h_triples"]) for r in records)


def count_predicates(records: list[dict]) -> Counter:
    return Counter(tr.get("predicate") for r in records for tr in r["t_triples"] + r["h_triples"])


open_records = load_jsonl(open_file)
closed_records = load_jsonl(closed_file)
closed_pruned = load_jsonl(closed_pruned_file)

summary_rows = [
    ("Open extraction", open_records),
    ("Closed schema", closed_records),
    ("Closed schema pruned", closed_pruned),
]

print(f"{'Pipeline':25} {'Pairs':>6} {'t_has':>6} {'h_has':>6} {'both':>6} {'Triples':>9}")
for name, recs in summary_rows:
    cov = coverage(recs)
    print(f"{name:25} {cov['total']:>6} {cov['t_has_triples']:>6} {cov['h_has_triples']:>6} {cov['both']:>6} {count_triples(recs):>9}")

print()
print("Open extraction top predicates:")
for pred, count in count_predicates(open_records).most_common(10):
    print(f"  {pred}: {count}")
print()
print("Closed schema top predicates:")
for pred, count in count_predicates(closed_records).most_common(10):
    print(f"  {pred}: {count}")
print()
print("Closed schema pruned top predicates:")
for pred, count in count_predicates(closed_pruned).most_common(10):
    print(f"  {pred}: {count}")

Pipeline                   Pairs  t_has  h_has   both   Triples
Open extraction              131    131    131    131      1118
Closed schema                131    131    131    131      1226
Closed schema pruned         131    125    126    122       962

Open extraction top predicates:
  is: 60
  of: 46
  said: 38
  described_as: 29
  in: 24
  located_in: 21
  was: 20
  for: 19
  occurred_in: 17
  from: 14

Closed schema top predicates:
  in: 146
  said: 111
  was: 110
  is: 104
  of: 103
  described_as: 60
  has: 58
  for: 51
  to: 44
  not_allow: 39

Closed schema pruned top predicates:
  in: 139
  was: 102
  is: 95
  of: 94
  said: 85
  has: 56
  described_as: 49
  for: 45
  not_allow: 36
  by: 33


In [62]:
for r_open, r_closed, r_pruned in zip(open_records[:3], closed_records[:3], closed_pruned[:3]):
    print(f"type={r_open['type']}")
    print(f"  t: {r_open['t_text']}")
    print(f"  h: {r_open['h_text']}")
    print(f"  open           t_triples: {r_open['t_triples']}")
    print(f"  open           h_triples: {r_open['h_triples']}")
    print(f"  closed         t_triples: {r_closed['t_triples']}")
    print(f"  closed         h_triples: {r_closed['h_triples']}")
    print(f"  closed pruned  t_triples: {r_pruned['t_triples']}")
    print(f"  closed pruned  h_triples: {r_pruned['h_triples']}")
    print("─" * 100)

type=negation
  t: Tariq Aziz was not considered a member of Saddam's innermost circle.
  h: Tariq Aziz was in Saddam's inner circle.
  open           t_triples: [{'subject': 'Tariq Aziz', 'predicate': 'not_considered', 'object': 'member'}, {'subject': 'member', 'predicate': 'of', 'object': 'Saddam'}, {'subject': 'member', 'predicate': 'described_as', 'object': 'innermost circle'}]
  open           h_triples: [{'subject': 'Tariq Aziz', 'predicate': 'was_in', 'object': 'inner circle'}, {'subject': 'inner circle', 'predicate': 'belongs_to', 'object': 'Saddam'}]
  closed         t_triples: [{'subject': 'Tariq Aziz', 'predicate': 'was', 'object': 'member'}, {'subject': 'member', 'predicate': 'of', 'object': "Saddam's circle"}, {'subject': "Saddam's circle", 'predicate': 'described_as', 'object': 'innermost'}, {'subject': 'Tariq Aziz', 'predicate': 'not_allow', 'object': 'considered'}]
  closed         h_triples: [{'subject': 'Tariq Aziz', 'predicate': 'was', 'object': "in Saddam's inner ci